## PURCHASES
![image_1770576617532.png](./image_1770576617532.png "image_1770576617532.png") 

In [0]:
%sql
select * from vendor_performance.bronze.purchases where VendorNumber = 4466

## PURCHASE PRICES
![image_1770576666471.png](./image_1770576666471.png "image_1770576666471.png")

In [0]:
%sql
select * from vendor_performance.bronze.purchase_prices where VendorNumber = 4466

## VENDOR INVOICE
![image_1770576836415.png](./image_1770576836415.png "image_1770576836415.png")

In [0]:
%sql
select * from vendor_performance.bronze.vendor_invoice where VendorNumber = 4466

## SALES
![image_1770576872833.png](./image_1770576872833.png "image_1770576872833.png")

In [0]:
%sql
select * from vendor_performance.bronze.sales where VendorNo = 4466

In [0]:
%sql
select 
  brand,
  purchaseprice,
  sum(quantity) as total_quantity,
  sum(dollars) as total_dollars
from vendor_performance.bronze.purchases
where VendorNumber = 4466
group by 1,2
order by total_quantity desc;

In [0]:
%sql
-- select 
--   p.brand,
--   p.purchaseprice,
--   sum(p.quantity) as total_quantity,
--   sum(p.dollars) as total_purchase_dollars,
--   sum(s.SalesQuantity) as total_sales_qty,
--   sum(s.SalesDollars) as total_sales_dollars,
--   sum(s.SalesPrice) as avg_sales_price 
-- from vendor_performance.bronze.purchases as p
-- join vendor_performance.bronze.sales as s 
-- on s.VendorNo = p.VendorNumber
-- where p.VendorNumber = 4466
-- group by 1,2
-- order by total_quantity desc;

select 
  Brand,
  sum(SalesQuantity) as total_sales_qty,
  sum(SalesDollars) as total_sales_dollars,
  sum(SalesPrice) as avg_sales_price 
from vendor_performance.bronze.sales 
where VendorNo = 4466
group by 1
order by 2 desc;

In [0]:
freight_summary = spark.sql("""
                            select 
                                VendorNumber, 
                                round(sum(Freight),2) as total_freight_cost
                            from vendor_performance.bronze.vendor_invoice
                            group by VendorNumber
                        """)
display(freight_summary)

In [0]:
%sql
select 
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    p.PurchasePrice,
    pp.Volume,
    pp.Price as ActualPrice,
    SUM(p.Quantity) as TotalPurchaseQty,
    SUM(p.Dollars) as TotalPurchaseDollars
from vendor_performance.bronze.purchases as p
join vendor_performance.bronze.purchase_prices as pp
on p.Brand = pp.Brand
where p.PurchasePrice > 0
group by 1,2,3,4,5,6
order by 8 


In [0]:
%sql
select 
  s.VendorNo,
  s.Brand,
  SUM(s.SalesDollars) as TotalSalesDollars,
  SUM(s.SalesPrice) as TotalSalesPrice,
  SUM(s.SalesQuantity) as TotalSalesQuantity,
  SUM(s.ExciseTax) as TotalExciseTax
from vendor_performance.bronze.sales as s
group by 1,2
order by 3

In [0]:
%sql
select 
  pp.VendorNumber,
  pp.Brand,
  pp.Price as ActualPrice,
  pp.PurchasePrice,
  SUM(s.SalesDollars) as TotalSalesDollars,
  SUM(s.SalesPrice) as TotalSalesPrice,
  SUM(s.SalesQuantity) as TotalSalesQuantity,
  SUM(s.ExciseTax) as TotalExciseTax,
  SUM(vi.Quantity) as TotalPurchaseQuantity,
  SUM(vi.Dollars) as TotalPurchaseDollars,
  SUM(vi.Freight) as TotalFreightCost
from vendor_performance.bronze.purchase_prices as pp
join vendor_performance.bronze.sales as s
on s.Brand = pp.Brand and s.VendorNo = pp.VendorNumber  
join vendor_performance.bronze.vendor_invoice as vi
on vi.VendorNumber = pp.VendorNumber
group by 1,2,3,4

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW temp_sales_summary AS
WITH FreightSummary AS (
    SELECT 
        VendorNumber,
        sum(Freight) as FreightCost
    from vendor_performance.bronze.vendor_invoice
    group by VendorNumber
),
PurchaseSummary AS (
    select 
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Price as ActualPrice,
        pp.Volume,
        sum(p.Quantity) as TotalPurchaseQuantity,
        sum(p.Dollars) as TotalPurchaseDollars
    from vendor_performance.bronze.purchases as p
    join vendor_performance.bronze.purchase_prices as pp
    on p.Brand = pp.Brand
    where p.PurchasePrice > 0
    group by 1,2,3,4,5,6,7
),
SalesSummary AS (
  select 
    VendorNo,
    Brand,
    sum(SalesDollars) as TotalSalesDollars,
    sum(SalesPrice) as TotalSalesPrice,
    sum(SalesQuantity) as TotalSalesQuantity,
    sum(ExciseTax) as TotalExciseTax
  from vendor_performance.bronze.sales as s
  group by 1,2
)
select 
  ps.VendorNumber,
  ps.VendorName,
  ps.Brand,
  ps.Description,
  ps.PurchasePrice,
  ps.ActualPrice,
  ps.Volume,
  ps.TotalPurchaseQuantity,
  ps.TotalPurchaseDollars,
  ss.TotalSalesQuantity,
  ss.TotalSalesPrice,
  ss.TotalSalesDollars,
  ss.TotalExciseTax,
  fs.FreightCost
from PurchaseSummary as ps
join SalesSummary as ss
on ps.Brand = ss.Brand and ps.VendorNumber = ss.VendorNo
join FreightSummary as fs
on ps.VendorNumber = fs.VendorNumber
order by ps.TotalPurchaseDollars desc


In [0]:
%sql
select count(*) from temp_sales_summary

In [0]:
dfgold = spark.sql("select * from temp_sales_summary")

In [0]:
from pyspark.sql.functions import col, sum
def check_missing(df):
    total_rows = df.count()
    null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    
    # Create a summary list
    summary_data = [{"Column": k, "Missing_Values": v, "Percentage": (v/total_rows)*100} 
                    for k, v in null_counts.items()]
    
    return spark.createDataFrame(summary_data)

display(check_missing(dfgold))

In [0]:
dfgold.printSchema()

In [0]:
from pyspark.sql.types import IntegerType, DecimalType
from pyspark.sql.functions import col, trim

df_type_casting = dfgold.withColumn("Volume", col("Volume").cast("float"))
df_filled_na = df_type_casting.na.fill(value=0, subset=["TotalSalesQuantity", "TotalSalesDollars", "TotalSalesPrice", "TotalExciseTax"])
df_no_whitespaces = df_filled_na.withColumn("VendorName", trim(col("VendorName")))

# df_with_spaces = dfgold.filter(col("VendorName").rlike(r"\s"))
display(df_no_whitespaces)

In [0]:
# Save the DataFrame as a managed Delta table
df_no_whitespaces.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("vendor_performance.silver.vendor_sales_summary")